In [53]:
import os, re, requests, math
from dotenv import load_dotenv
from openai import AzureOpenAI
import json, time, yaml
from pathlib import Path

CONFIG_PATH = Path("config.yaml")
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

load_dotenv(config["paths"]["load_dotenv"])
client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_API_BASE"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
)
DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"]  # deployment name, NOT the model name

In [54]:
def llm(messages, temperature=0):
    """One call to the model. Takes a list of messages, returns the reply text.
    """
    resp = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

In [55]:
def get_data(url, **params):
    resp = requests.get(url, params=params, headers={"User-Agent": config["http"]["user_agent"]},
                        timeout=config["http"]["timeout"])
    resp.raise_for_status()
    return resp.json()

In [56]:
# WORKING MEMORY
class WorkingMemory:
    def __init__(self, system):
        self.messages = [{"role": "system", "content": system}]
    def add(self, role, content):
        self.messages.append({"role": role, "content": content}) # append

In [57]:
# EPISODIC MEMORY
# timestamped, recalled by keywords
class EpisodicMemory:
    def __init__(self, path="episodes.json"):
        self.path = Path(path)
        self.episodes = json.loads(self.path.read_text()) if self.path.exists() else []
    def record(self, summary):
        self.episodes.append({"time": time.strftime("%Y-%m-%d %H:%M"), "event": summary})
        self.path.write_text(json.dumps(self.episodes, indent=2))
    def recall(self, keyword):
        return [e for e in self.episodes if keyword.lower() in e["event"].lower()]

In [58]:
# SEMANTIC MEMORY
# facts as key: value
class SemanticMemory:
    def __init__(self):
        self.facts = {}
    def learn(self, key, value):
        self.facts[key] = value
    def as_prompt(self):
        return "Known facts:\n" + "\n".join(f"{k}: {v}" for k, v in self.facts.items())

In [59]:
# PROCEDURAL MEMORY
# reusable instructions for how to behave or act
procedures = {
    "report_current_state": (
        "Step 1: Parse the user's location input and map it to its corresponding Singapore region "
        "('north', 'south', 'east', 'west', or 'central'). If the location is ambiguous or missing, "
        "default to the system's current location context or explicitly ask the user for clarification.\n"
        "Step 2: Concurrently or sequentially call the external API tools to gather the live Weather, "
        "Temperature, and 24-hour PSI/PM2.5 metrics for that specific region. If any critical API fails "
        "or times out, record a fallback failure state instead of inventing placeholder metrics.\n"
        "Step 3: Aggregate the data and evaluate it against standard safety thresholds (e.g., NEA PSI "
        "categories and heat stress rules). Synthesize a structural response containing the location "
        "timestamp, explicit readings, and a clear, deterministic activity recommendation based on the "
        "highest environmental risk factor found."
    ),
}

# Tool Functions
- Weather
- Temperature
- PSI readings

In [60]:
def fetch_psi() -> tuple[str, dict]:
    """Return (updated timestamp, {region: 24h PSI}) from NEA's real-time API."""
    item = get_data(config["urls"]["psi"])["data"]["items"][0]
    return item["updatedTimestamp"], item["readings"]["psi_twenty_four_hourly"]

def get_psi_readings() -> str:
    """Singapore's current 24-hour PSI by region (NEA). Current data, not a forecast. No arguments."""
    ts, psi = fetch_psi()
    return f"PSI for Singapore, updated {ts}:\n" + "\n".join(
        f"{r.capitalize()}: {v}" for r, v in psi.items())

In [61]:
print(get_psi_readings()[:200], "...\n")

PSI for Singapore, updated 2026-10-18T16:15:34+08:00:
North: 115
South: 148
West: 204
East: 150
Central: 158 ...



In [62]:
def fetch_pm25() -> tuple[str, dict]:
    """Return (updated timestamp, {region: 1hr PM2.5}) from NEA's real-time API."""
    item = get_data(config["urls"]["pm25"])["data"]["items"][0]
    return item["updatedTimestamp"], item["readings"]["pm25_one_hourly"]

def get_pm25_readings() -> str:
    """Singapore's current one-hour pm25 by region (NEA). Current data, not a forecast. No arguments."""
    tv, pm25 = fetch_pm25()
    return f"PM2.5 for Singapore, updated {tv}:\n" + "\n".join(
        f"{r.capitalize()}: {v}" for r, v in pm25.items())

In [63]:
print(get_pm25_readings()[:200], "...\n")

PM2.5 for Singapore, updated 2026-10-18T16:15:34+08:00:
North: 37
South: 69
West: 77
East: 70
Central: 58 ...



In [64]:
REGION_CENTRES = {r: tuple(c) for r, c in config["region_centres"].items()}
REGIONS = list(REGION_CENTRES)

def geocode(place: str, n: int=config["agent"]["n_candidates"]) -> list[dict]:
    """Top-n OneMap matches for a Singapore place: [{name, address, lat, lon}, ...]."""
    results = get_data(config["urls"]["onemap"], searchVal=place, returnGeom="Y", getAddrDetails="Y", pageNum=1)["results"]
    if not results:
        raise ValueError(f"No location found for {place!r}")
    return [{"name": r["SEARCHVAL"], "address": r["ADDRESS"], "lat": float(r["LATITUDE"]), "lon": float(r["LONGITUDE"])} 
            for r in results[:n]]

def nearest_region(lat: float, lon: float) -> str:
    return min(REGION_CENTRES, key=lambda r: math.dist((lat, lon), REGION_CENTRES[r]))

def psi_for_place(place: str) -> str:
    top = geocode(place)[0]
    lat, lon = top["lat"], top["lon"]
    region = nearest_region(lat, lon)
    ts, psi = fetch_psi()
    return f"{place} ({lat:.4f}, {lon:.4f}) -> {region} region: 24h PSI {psi[region]} (updated {ts})"

ntu = geocode("NTU")   # one OneMap call, reused below (avoids 429 Too Many Requests)
print(ntu)
print(nearest_region(ntu[0]["lat"], ntu[0]["lon"]))

[{'name': 'NANYANG TECHNOLOGICAL UNIVERSITY', 'address': '94 NANYANG CRESCENT NANYANG TECHNOLOGICAL UNIVERSITY SINGAPORE 637034', 'lat': 1.352949370203277, 'lon': 103.6892235086704}, {'name': 'NANYANG TECHNOLOGICAL UNIVERSITY', 'address': '71 NANYANG CRESCENT NANYANG TECHNOLOGICAL UNIVERSITY SINGAPORE 637035', 'lat': 1.354282743739479, 'lon': 103.6894395003443}, {'name': 'NANYANG TECHNOLOGICAL UNIVERSITY', 'address': '50 NANYANG AVENUE NANYANG TECHNOLOGICAL UNIVERSITY SINGAPORE 639798', 'lat': 1.343058936358113, 'lon': 103.6809726894903}, {'name': 'NTU INNOVATION CENTRE', 'address': '71 NANYANG DRIVE NTU INNOVATION CENTRE SINGAPORE 638075', 'lat': 1.348278894770232, 'lon': 103.6816475606479}, {'name': 'NTU@ONE-NORTH', 'address': '11 SLIM BARRACKS RISE NTU@ONE-NORTH SINGAPORE 138664', 'lat': 1.30376708827617, 'lon': 103.7899076195128}]
west


In [65]:
# resolving agent for geocode
def resolve_place(qns):
    place = llm([
        {"role": "system", "content": "Extract the Singapore place the user is asking about. Reply with only the place name."},
        {"role": "user", "content": qns},
    ]).strip()
    cands = geocode(place)   # raises ValueError if OneMap finds nothing
    listing = "\n".join(f"{i}: {c['name']} | {c['address']}" for i, c in enumerate(cands))

    reply = llm([
        {"role": "system", "content": "You match a place name to a list of map search results, so we can get coordinates for a weather and air-quality lookup. "
                                      "You are NOT answering the user's question; only decide which result is in the place they mean. "
                                      "A result matches if it IS that place, or is a building or location INSIDE that place "
                                      "(e.g. for a town or district, any location in that town counts). "
                                      "A result does NOT match if it only shares part of the name but is a different place. "
                                      "Reply with only the number of the first matching result, or -1 if none match."},
        {"role": "user", "content": f"Question: {qns}\nPlace: {place}\nCandidates:\n{listing}"},
    ]).strip()

    i = int(reply) if reply.lstrip("-").isdigit() else -1 # guardrail
    if not 0 <= i < len(cands):
        raise ValueError(f"No candidate matched {place!r}; picker replied {reply!r}")
    return {**cands[i], "query": place, "region": nearest_region(cands[i]["lat"], cands[i]["lon"])}

# verification
def verify_place(qns, p):
    reply = llm([
        {"role": "system", "content": "You check a location lookup for a weather and air-quality report, which only needs coordinates in the right area. "
                                      "Reply YES if the resolved location is the place the user asked about, or is a building or location INSIDE it "
                                      "(e.g. any location in a town counts for that town). "
                                      "Reply NO if it is a different place, even if the name is similar (e.g. NTUC instead of NTU), or is in a different area. "
                                      "Start with YES or NO, then a short reason."},
        {"role": "user", "content": f"Question: {qns}\nResolved to: {p['name']}, {p['address']}\n"
                                    "Is this the place the user meant?"},
    ])
    return reply.strip().upper().startswith("YES"), reply

In [66]:
PSI_BANDS = config["psi_bands"]   # {band: upper bound}, in order from config.yaml
def psi_band(v):
    return next(band for band, hi in PSI_BANDS.items() if v <= hi)

PM25_BANDS = config["pm25_bands"]   # {band: upper bound}, in order from config.yaml
def pm25_band(u):
    return next(band for band, hi in PM25_BANDS.items() if u <= hi)

PM25_ADVICE = {   # what to do NOW, by NEA 1-hr PM2.5 band ("normal" falls through to weather/heat)
    "very high": "Avoid outdoor activity; stay indoors.",
    "high":      "Minimise outdoor activity.",
    "elevated":  "Reduce prolonged or strenuous outdoor activity.",
}

In [67]:
def fetch_temperature(lat, lon):
    d = get_data(config["urls"]["temperature"])["data"]
    stations = {s["id"]: (s["location"]["latitude"], s["location"]["longitude"]) for s in d["stations"]}
    reading = d["readings"][0]
    values = {r["stationId"]: r["value"] for r in reading["data"]}
    nearest = min(values, key=lambda s: math.dist((lat, lon), stations[s]))
    return values[nearest], round(sum(values.values()) / len(values), 1), reading["timestamp"]

In [68]:
def fetch_forecast(lat, lon):
    d = get_data(config["urls"]["forecast"])["data"]
    areas = {a["name"]: (a["label_location"]["latitude"], a["label_location"]["longitude"]) for a in d["area_metadata"]}
    item = d["items"][0]
    fc = {f["area"]: f["forecast"] for f in item["forecasts"]}
    area = min(fc, key=lambda a: math.dist((lat, lon), areas[a]))
    return area, fc[area], item["valid_period"]["text"]

In [69]:
def recommend(psi, pm25, temp, forecast):
    # deterministic: highest risk wins. 1-hr PM2.5 decides what to do right now;
    # the 24-hr PSI adds a caveat, since it reflects longer exposure and lags behind.
    band = pm25_band(pm25)
    if band in PM25_ADVICE:
        advice = PM25_ADVICE[band]
    elif any(w in forecast.lower() for w in ("thunder", "rain", "showers")):
        advice = "Keep activities indoors or bring rain cover."
    elif temp >= 33:
        advice = "Limit strenuous activity in the heat and stay hydrated."
    else:
        advice = "Normal activities are fine."
    if psi > 100:   # 24-hr PSI unhealthy or worse
        advice += f" Caveat: 24-hr PSI is still {psi_band(psi)}, so check again before heading out."
    return advice

def gather(p):
    ts, psi = fetch_psi()
    tv, pm25 = fetch_pm25()
    temp, island_avg, _ = fetch_temperature(p["lat"], p["lon"])
    area, forecast, period = fetch_forecast(p["lat"], p["lon"])
    v, u = psi[p["region"]], pm25[p["region"]]
    return {"location": p["name"], "region": p["region"], "updated": ts,
            "psi_24h": v, "psi_band": psi_band(v),
            "pm25_1h": u, "pm25_band": pm25_band(u),
            "temp_c": temp, "island_avg_c": island_avg, "temp_diff_c": round(temp - island_avg, 1),
            "forecast_area": area, "forecast": forecast, "forecast_period": period,
            "recommendation": recommend(v, u, temp, forecast)}

In [ ]:
# report generation and self-checking
MAX_TRIES = config["agent"]["max_tries"]   # 1 first draft + up to (MAX_TRIES - 1) rewrites
REPORT_TEMPLATE = (
    "Last updated: <time> for <location> (<region> region)\n"
    "PSI (24-hr): <value> (<band>)\n"
    "PM 2.5 (hourly): <value> (<band>)\n"
    "Temperature: <temp>°C (<diff>°C higher/lower than island average)\n"
    "Forecast (<period>, <area>): <forecast>\n"
    "Recommendation: <recommendation>"
)

# prefixes must match write_report's template exactly; rename a template line -> update here too
REPORT_LINES = {"psi": "PSI (24-hr):", "pm25": "PM 2.5 (hourly):", "temp": "Temperature:",
                "forecast": "Forecast", "recommendation": "Recommendation:"}
assert all(prefix in REPORT_TEMPLATE for prefix in REPORT_LINES.values()), "REPORT_LINES doesn't match REPORT_TEMPLATE"

class ReportCheckError(Exception):
    """Raised when the writer can't produce a report that matches the facts within MAX_TRIES."""
    def __init__(self, problems, draft):
        super().__init__("Report still failed checks: " + "; ".join(problems))
        self.problems = problems   # what was still wrong
        self.draft = draft         # last draft, kept for debugging

def write_report(facts, draft=None, problems=None):
    msgs = [
        {"role": "system", "content": procedures["report_current_state"] +
            "\nUse exactly this format, one item per line:\n" + REPORT_TEMPLATE},
        {"role": "user", "content": json.dumps(facts)},
    ]
    if problems:
        # show the model its own draft, so it fixes it instead of starting from scratch
        msgs.append({"role": "assistant", "content": draft})
        msgs.append({"role": "user", "content": "Your draft above had these problems: " + "; ".join(problems)
                                                + ". Reply with the full corrected report."})
    return llm(msgs)

def has_number(text, value):
    """True if value appears as a whole number in text, so 21 doesn't match inside 218."""
    forms = {str(value)}
    if isinstance(value, float) and value.is_integer():
        forms.add(str(int(value)))   # 27.0 may be written as 27
    return any(re.search(rf"(?<![\d.]){re.escape(f)}(?!\.?\d)", text) for f in forms)

def line_for(report, prefix):
    """Text after `prefix` on the report line that starts with it, or None if that line is missing.
    Only the part after the label is returned, so the 24 in 'PSI (24-hr)' can't count as a reading."""
    for line in report.splitlines():
        if line.strip().lower().startswith(prefix.lower()):
            return line.strip()[len(prefix):]
    return None

def check_report(report, facts):
    problems = []
    lines = {key: line_for(report, prefix) for key, prefix in REPORT_LINES.items()}
    for key, text in lines.items():
        if text is None:
            problems.append(f"missing line {REPORT_LINES[key]!r}")
    psi, pm25, temp, fc, rec = (lines[k] or "" for k in REPORT_LINES)   # "" -> its checks below fail too

    # each value is checked on ITS OWN line, so swapping the PSI and PM2.5 numbers is caught
    if lines["psi"] is not None:
        if not has_number(psi, facts["psi_24h"]):
            problems.append(f"PSI line should show {facts['psi_24h']}")
        # band in brackets, so "unhealthy" doesn't match inside "very unhealthy"
        if f"({facts['psi_band']})" not in psi.lower():
            problems.append(f"PSI band should be ({facts['psi_band']})")
    if lines["pm25"] is not None:
        if not has_number(pm25, facts["pm25_1h"]):
            problems.append(f"PM 2.5 line should show {facts['pm25_1h']}")
        # same trick: "(high)" doesn't match inside "(very high)"
        if f"({facts['pm25_band']})" not in pm25.lower():
            problems.append(f"PM 2.5 band should be ({facts['pm25_band']})")
    if lines["temp"] is not None:
        if not has_number(temp, facts["temp_c"]):
            problems.append(f"Temperature line should show {facts['temp_c']}")
        if not has_number(temp, abs(facts["temp_diff_c"])):   # abs: the template says higher/lower, not +/-
            problems.append(f"Temperature difference should be {abs(facts['temp_diff_c'])}")
    if lines["forecast"] is not None and facts["forecast"].lower() not in fc.lower():
        problems.append(f"Forecast line should say {facts['forecast']!r}")
    if lines["recommendation"] is not None and facts["recommendation"].lower() not in rec.lower():
        problems.append(f"Recommendation must be exactly {facts['recommendation']!r}")
    return problems

def write_checked_report(facts, max_tries=MAX_TRIES):
    """Write -> check -> rewrite with feedback, until the report passes or tries run out."""
    report, problems = None, None
    for _ in range(max_tries):
        report = write_report(facts, report, problems)
        problems = check_report(report, facts)
        if not problems:
            return report
    raise ReportCheckError(problems, report)

def answer(qns):
    p = resolve_place(qns) # obtain the results from the given question
    ok, why = verify_place(qns, p) # cross-check the returned results from facts
    if not ok:
        return f"I'm not sure which place you mean ({p['name']}?). {why}"
    facts = gather(p)
    return write_checked_report(facts)

In [ ]:
# checking if 'problems' work with a bad input
bad = report.replace(str(facts["psi_24h"]), "999")
problems = check_report(bad, facts)
print(problems)

['PSI line should show 154']


In [76]:
qns = "What is the status like in tampines"
# location mapping resolve
p = resolve_place(qns)
print(p)
# verification
ok, why = verify_place(qns, p)
print(ok, "|", why)
# fact check against return results
facts = gather(p)
print(json.dumps(facts, indent=2))
# report
report = write_checked_report(facts) 
print("report:\n" + report)
print("  check:", check_report(report, facts) or "passed")

{'name': 'TAMPINES 9 BUILDING', 'address': '9 TAMPINES STREET 92 TAMPINES 9 BUILDING SINGAPORE 528871', 'lat': 1.343330328726356, 'lon': 103.9443723970848, 'query': 'Tampines', 'region': 'east'}
True | YES, the resolved location is within Tampines, so it matches the user's query about Tampines.
{
  "location": "TAMPINES 9 BUILDING",
  "region": "east",
  "updated": "2026-10-18T16:30:35+08:00",
  "psi_24h": 150,
  "psi_band": "unhealthy",
  "pm25_1h": 70,
  "pm25_band": "elevated",
  "temp_c": 30.2,
  "island_avg_c": 31.5,
  "temp_diff_c": -1.3,
  "forecast_area": "Tampines",
  "forecast": "Partly Cloudy (Day)",
  "forecast_period": "4.30 pm to 6.30 pm",
  "recommendation": "Reduce prolonged or strenuous outdoor activity. Caveat: 24-hr PSI is still unhealthy, so check again before heading out."
}
report:
Last updated: 2026-10-18T16:30:35+08:00 for TAMPINES 9 BUILDING (east region)
PSI (24-hr): 150 (unhealthy)
PM 2.5 (hourly): 70 (elevated)
Temperature: 30.2°C (1.3°C lower than island av

In [78]:
answer(qns)

'Last updated: 2026-10-18T17:30:39+08:00 for TAMPINES 9 BUILDING (east region)\nPSI (24-hr): 146 (unhealthy)\nPM 2.5 (hourly): 82 (elevated)\nTemperature: 29.4°C (1.2°C lower than island average)\nForecast (5.00 pm to 7.00 pm, Tampines): Partly Cloudy (Day)\nRecommendation: Reduce prolonged or strenuous outdoor activity. Caveat: 24-hr PSI is still unhealthy, so check again before heading out.'